In [2]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/mujtaba-sts-dataset/SD-Tooth.zip.003
/kaggle/input/mujtaba-sts-dataset/SD-Tooth.zip.013
/kaggle/input/mujtaba-sts-dataset/SD-Tooth.zip.015
/kaggle/input/mujtaba-sts-dataset/SD-Tooth.zip.006
/kaggle/input/mujtaba-sts-dataset/SD-Tooth.zip.005
/kaggle/input/mujtaba-sts-dataset/SD-Tooth.zip.002
/kaggle/input/mujtaba-sts-dataset/SD-Tooth.zip.011
/kaggle/input/mujtaba-sts-dataset/SD-Tooth.zip.004
/kaggle/input/mujtaba-sts-dataset/SD-Tooth.zip.009
/kaggle/input/mujtaba-sts-dataset/SD-Tooth.zip.014
/kaggle/input/mujtaba-sts-dataset/SD-Tooth.zip.001
/kaggle/input/mujtaba-sts-dataset/SD-Tooth.zip.008
/kaggle/input/mujtaba-sts-dataset/SD-Tooth.zip.012
/kaggle/input/mujtaba-sts-dataset/SD-Tooth.zip.010
/kaggle/input/mujtaba-sts-dataset/SD-Tooth.zip.007


In [3]:
import os

# Define paths - using /kaggle/temp for space
input_dir = "/kaggle/input/mujtaba-sts-dataset" 
if not os.path.exists(input_dir):
    input_dir = "/kaggle/input/datasets/mujtabajunaid/mujtaba-sts-dataset"

combined_zip = "/kaggle/temp/SD-Tooth-Full.zip"
extract_path = "/kaggle/temp/sts_extracted"

# Ensure the temp directory exists
os.makedirs(extract_path, exist_ok=True)

print(f"--- Step 1: Combining 15 parts from {input_dir} ---")
!cat {input_dir}/SD-Tooth.zip.* > {combined_zip}

print("\n--- Step 2: Verifying combined file size ---")
!du -sh {combined_zip}

print("\n--- Step 3: Extracting dataset to /kaggle/temp ---")
!unzip -q {combined_zip} -d {extract_path}

print("\n--- Step 4: Cleanup ---")
!rm /kaggle/temp/SD-Tooth-Full.zip
print("Zip file deleted. Space cleared, extracted data remains.")

# ==================================================================

--- Step 1: Combining 15 parts from /kaggle/input/mujtaba-sts-dataset ---

--- Step 2: Verifying combined file size ---
30G	/kaggle/temp/SD-Tooth-Full.zip

--- Step 3: Extracting dataset to /kaggle/temp ---

--- Step 4: Cleanup ---
Zip file deleted. Space cleared, extracted data remains.


In [4]:
!pip install -q "monai>=1.5.0" einops

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 29.5 MB/s eta 0:00:00a 0:00:01


In [ ]:
import os

def list_folders(startpath, max_depth=3):
    for root, dirs, files in os.walk(startpath):
        level = root.replace(startpath, '').count(os.sep)
        if level < max_depth:
            indent = ' ' * 4 * (level)
            print(f'{indent}{os.path.basename(root)}/')
            subindent = ' ' * 4 * (level + 1)
            # Just print the first 2 files to see naming convention
            for f in files[:2]:
                print(f'{subindent}{f}')

list_folders(extract_path)

In [ ]:
import os
import glob

# Use a recursive search to find ALL png files
all_files = glob.glob("/kaggle/temp/sts_extracted/**/*.png", recursive=True)
all_3d_files = glob.glob("/kaggle/temp/sts_extracted/**/*.nii.gz", recursive=True)

print(f"Total PNGs found: {len(all_files)}")
print(f"Total NIfTIs found: {len(all_3d_files)}")

# Print the first 10 paths to see the actual naming convention
if all_files:
    print("\nSample PNG Paths:")
    for f in all_files[:10]:
        print(f)

In [ ]:
import glob
import os
import pandas as pd

BASE_2D = "/kaggle/temp/sts_extracted/SD-Tooth/STS-2D-Tooth"

# Find all PNGs in Labeled folders
all_labeled_images = sorted(glob.glob(f"{BASE_2D}/**/Labeled/Image/*.png", recursive=True))
all_labeled_masks = sorted(glob.glob(f"{BASE_2D}/**/Labeled/Mask/*.png", recursive=True))

# 1. Check counts
print(f"📊 Total Labeled Images: {len(all_labeled_images)}")
print(f"📊 Total Labeled Masks: {len(all_labeled_masks)}")

# 2. Verify Filename Matching
mismatches = []
for img, msk in zip(all_labeled_images, all_labeled_masks):
    if os.path.basename(img) != os.path.basename(msk):
        mismatches.append((img, msk))

if len(mismatches) == 0 and len(all_labeled_images) > 0:
    print("✅ Success: All filenames match perfectly.")
else:
    print(f"❌ Error: Found {len(mismatches)} filename mismatches!")
    if mismatches: print(f"Sample Mismatch: {mismatches[0]}")

In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np
import random
import os

def visual_sanity_check(img_paths, msk_paths, n=3):
    indices = random.sample(range(len(img_paths)), n)
    plt.figure(figsize=(16, 8))
    
    for i, idx in enumerate(indices):
        # Load as Grayscale
        img = cv2.imread(img_paths[idx], cv2.IMREAD_GRAYSCALE)
        msk = cv2.imread(msk_paths[idx], cv2.IMREAD_GRAYSCALE)
        
        # Ensure they are the same size for the overlay
        if img.shape != msk.shape:
            msk = cv2.resize(msk, (img.shape[1], img.shape[0]))

        # Create Color Overlay
        color_msk = np.zeros((img.shape[0], img.shape[1], 3), dtype=np.uint8)
        color_msk[msk > 0] = [0, 255, 0] # Green for teeth
        
        # Blend
        img_bgr = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
        blended = cv2.addWeighted(img_bgr, 0.7, color_msk, 0.3, 0)
        
        plt.subplot(1, n, i+1)
        plt.imshow(cv2.cvtColor(blended, cv2.COLOR_BGR2RGB))
        plt.title(f"Check: {os.path.basename(img_paths[idx])}")
        plt.axis('off')
    plt.show()

visual_sanity_check(all_labeled_images, all_labeled_masks)

In [ ]:
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, 
    ScaleIntensityRanged, Resized, NormalizeIntensityd, EnsureTyped
)
from monai.data import CacheDataset, DataLoader

# Build the pipeline mentioned in the paper
train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    
    # 1. Intensity Clipping: The 'Dental Window'
    # Removes high-intensity metal (fillings) and low-intensity air
    ScaleIntensityRanged(
        keys=["image"], a_min=-100, a_max=1500,
        b_min=0.0, b_max=1.0, clip=True,
    ),
    
    # 2. Resizing to Paper Standard (640x320 for Panoramic)
    Resized(keys=["image", "label"], spatial_size=(640, 320), mode=("bilinear", "nearest")),
    
    # 3. Z-Score Normalization
    NormalizeIntensityd(keys=["image"], nonzero=True),
    EnsureTyped(keys=["image", "label"]),
])

# Create Dataset Pairs
data_dicts = [{"image": img, "label": msk} for img, msk in zip(all_labeled_images, all_labeled_masks)]

# 85/15 Split
split_idx = int(len(data_dicts) * 0.85)
train_files, val_files = data_dicts[:split_idx], data_dicts[split_idx:]

# Initialize CacheDataset (Uses Kaggle RAM)
train_ds = CacheDataset(data=train_files, transform=train_transforms, cache_rate=1.0)
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=2)

val_ds = CacheDataset(data=val_files, transform=train_transforms, cache_rate=1.0)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False)

In [ ]:
import torch
import torch.nn as nn
from monai.losses import DiceCELoss

# --- STEP 1: Define Building Blocks (Must come first) ---

class Recurrent_block(nn.Module):
    def __init__(self, ch_out, t=2):
        super().__init__()
        self.t = t
        self.conv = nn.Sequential(
            nn.Conv2d(ch_out, ch_out, kernel_size=3, padding=1),
            nn.BatchNorm2d(ch_out),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        for i in range(self.t):
            if i == 0:
                g = self.conv(x)
            else:
                g = self.conv(x + g)
        return g

class RRCNN_block(nn.Module):
    def __init__(self, ch_in, ch_out, t=2):
        super().__init__()
        self.RCNN = nn.Sequential(
            Recurrent_block(ch_out, t=t),
            Recurrent_block(ch_out, t=t)
        )
        self.Conv_1x1 = nn.Conv2d(ch_in, ch_out, kernel_size=1)
        
    def forward(self, x):
        x = self.Conv_1x1(x)
        return x + self.RCNN(x) # Residual Identity Mapping

# --- STEP 2: Define Full R2U-Net Assembly ---

class R2UNet(nn.Module):
    def __init__(self, img_ch=3, output_ch=1, t=2):
        super(R2UNet, self).__init__()
        
        self.Maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.Upsample = nn.UpsamplingBilinear2d(scale_factor=2)

        # Encoder (Downsampling)
        self.RRCNN1 = RRCNN_block(img_ch, 64, t=t)
        self.RRCNN2 = RRCNN_block(64, 128, t=t)
        self.RRCNN3 = RRCNN_block(128, 256, t=t)
        
        # Decoder (Upsampling)
        self.Up3 = RRCNN_block(256 + 128, 128, t=t)
        self.Up2 = RRCNN_block(128 + 64, 64, t=t)
        
        self.Conv_1x1 = nn.Conv2d(64, output_ch, kernel_size=1, stride=1, padding=0)

    def forward(self, x):
        # Encoding path
        e1 = self.RRCNN1(x)
        
        e2 = self.Maxpool(e1)
        e2 = self.RRCNN2(e2)
        
        e3 = self.Maxpool(e2)
        e3 = self.RRCNN3(e3)

        # Decoding path with Skip Connections
        d3 = self.Upsample(e3)
        d3 = torch.cat((e2, d3), dim=1) # Concatenation bridge
        d3 = self.Up3(d3)

        d2 = self.Upsample(d3)
        d2 = torch.cat((e1, d2), dim=1) # Concatenation bridge
        d2 = self.Up2(d2)

        out = self.Conv_1x1(d2)
        return torch.sigmoid(out)

# --- STEP 3: Initialization & Training Loop ---

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize with img_ch=3 because your PNGs are being loaded as RGB
model = R2UNet(img_ch=3, output_ch=1, t=2).to(device)

# Loss Function Note: 
# We set sigmoid=False because the model already has a sigmoid layer at the end.
loss_function = DiceCELoss(sigmoid=False)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
import gc

# 1. Clear everything before starting
gc.collect()
torch.cuda.empty_cache()

# 2. Re-initialize model with lower memory footprint if needed
model = R2UNet(img_ch=3, output_ch=1, t=2).to(device)

# 3. Training Parameters
accumulation_steps = 4  # Simulate batch size of 4 while processing 1 image
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_function = DiceCELoss(sigmoid=False)

print("🚀 Starting Memory-Efficient Training...")

for epoch in range(20):
    model.train()
    epoch_loss = 0
    optimizer.zero_grad()
    
    for i, batch_data in enumerate(train_loader):
        inputs = batch_data["image"].to(device)
        labels = batch_data["label"].to(device)
        
        # Forward pass
        outputs = model(inputs)
        loss = loss_function(outputs, labels)
        
        # Scale the loss by accumulation steps
        loss = loss / accumulation_steps
        loss.backward()
        
        # Only update weights every 'accumulation_steps'
        if (i + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()
            
        epoch_loss += loss.item() * accumulation_steps
        
        # 4. CRITICAL: Delete tensors and clear cache to free up RAM
        del inputs, labels, outputs
        if i % 10 == 0:
            torch.cuda.empty_cache()
            gc.collect()
            
    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch+1}/20 | Avg Loss: {avg_loss:.4f}")

# Save the model
torch.save(model.state_dict(), "r2unet_dental_fixed.pth")

In [ ]:
import torch.cuda.amp as amp # Added for Mixed Precision
import gc

# 1. Reset GPU Cache completely
gc.collect()
torch.cuda.empty_cache()

# 2. Re-initialize with a smaller batch_size in your DataLoader (set to 1 or 2)
# model = R2UNet(img_ch=3, output_ch=1, t=2).to(device)

scaler = amp.GradScaler() # Handles the float16/float32 scaling
accumulation_steps = 8    # High accumulation to keep effective batch size high

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_function = DiceCELoss(sigmoid=False)

print("🚀 Starting Nuclear Memory-Efficient Training...")

for epoch in range(20):
    model.train()
    epoch_loss = 0
    optimizer.zero_grad()
    
    for i, batch_data in enumerate(train_loader):
        inputs = batch_data["image"].to(device)
        labels = batch_data["label"].to(device)
        
        # 3. Use Autocast for Mixed Precision
        with amp.autocast():
            outputs = model(inputs)
            loss = loss_function(outputs, labels)
            loss = loss / accumulation_steps
        
        # 4. Scaled Backward Pass
        scaler.scale(loss).backward()
        
        if (i + 1) % accumulation_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            
        epoch_loss += loss.item() * accumulation_steps
        
        # Manually clear memory
        del inputs, labels, outputs
        if i % 5 == 0:
            torch.cuda.empty_cache()
            
    print(f"Epoch {epoch+1} | Loss: {epoch_loss/len(train_loader):.4f}")

torch.save(model.state_dict(), "r2unet_final_optimized.pth")

In [ ]:
import matplotlib.pyplot as plt

model.eval()
with torch.no_grad():
    # Grab one batch from val_loader
    batch = next(iter(val_loader))
    test_input = batch["image"].to(device)
    test_label = batch["label"].to(device)
    
    output = model(test_input)
    # Threshold at 0.5 to get binary mask
    pred = (output > 0.5).float()

    plt.figure(figsize=(12, 6))
    plt.subplot(1, 3, 1); plt.imshow(test_input[0, 0].cpu(), cmap='gray'); plt.title("Original X-ray")
    plt.subplot(1, 3, 2); plt.imshow(test_label[0, 0].cpu(), cmap='gray'); plt.title("Ground Truth")
    plt.subplot(1, 3, 3); plt.imshow(pred[0, 0].cpu(), cmap='gray'); plt.title("Model Prediction")
    plt.show()

In [ ]:
from monai.metrics import DiceMetric, HausdorffDistanceMetric

# 1. Initialize metrics
dice_metric = DiceMetric(include_background=False, reduction="mean")
hd95_metric = HausdorffDistanceMetric(percentile=95, include_background=False)

model.eval()
with torch.no_grad():
    for val_data in val_loader:
        val_inputs, val_labels = val_data["image"].to(device), val_data["label"].to(device)
        val_outputs = model(val_inputs)
        
        # Binarize outputs (0 or 1)
        val_outputs = (val_outputs > 0.5).float()
        
        # 2. Compute metrics
        dice_metric(y_pred=val_outputs, y=val_labels)
        hd95_metric(y_pred=val_outputs, y=val_labels)

    # 3. Aggregate and Print
    print(f"Final Mean Dice: {dice_metric.aggregate().item():.4f}")
    print(f"Final HD95: {hd95_metric.aggregate().item():.4f} mm")
    
    # Reset for next epoch
    dice_metric.reset()
    hd95_metric.reset()

# Approach 2

In [5]:
"""
=============================================================================
  DENTAL SEGMENTATION - PHASE 1: TEACHER MODEL (Swin UNETR)
  Based on: STS-Tooth (2025) - Semi-Supervised Tooth Segmentation Dataset
  Architecture: Swin UNETR (2D) via MONAI
  Target: Kaggle T4/P100 (~14.5 GB VRAM)
=============================================================================
  MEMORY STRATEGY SUMMARY:
  - AMP (fp16) via torch.cuda.amp
  - Gradient Accumulation (effective batch = ACCUM_STEPS * BATCH_SIZE)
  - Single-channel grayscale input (not 3-ch RGB) to cut VRAM ~3x
  - CacheDataset only for labeled set (900 images) — fits in Kaggle RAM
  - Aggressive gc + cuda cache clearing every N steps
  - img_ch=1, spatial_size=(640,320) — Swin UNETR window tiles fit cleanly
=============================================================================
"""

# ─────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
import os
import gc
import glob
import random
import logging
import warnings
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

import monai
from monai.config import print_config
from monai.data import CacheDataset, DataLoader, decollate_batch
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric, HausdorffDistanceMetric
from monai.networks.nets import SwinUNETR
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    ScaleIntensityRanged,
    Resized,
    NormalizeIntensityd,
    EnsureTyped,
    RandFlipd,
    RandRotate90d,
    RandShiftIntensityd,
    RandGaussianNoised,
    RandAdjustContrastd,
    AsDiscreted,
    Activationsd,
)
from monai.utils import set_determinism

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)

# ─────────────────────────────────────────────────────────────────────────────
# 1.  CONFIGURATION — all tunable knobs in one place
# ─────────────────────────────────────────────────────────────────────────────
CFG = {
    # ── Dataset paths ──────────────────────────────────────────────────────
    "data_root":        "/kaggle/temp/sts_extracted/SD-Tooth",          # Added SD-Tooth here
    
    "labeled_img_dir":  "STS-2D-Tooth/A-PXI/Labeled/Image",             # Capital I, singular
    "labeled_msk_dir":  "STS-2D-Tooth/A-PXI/Labeled/Mask",              # Capital M, singular
    
    # Child PXI
    "child_img_dir":    "STS-2D-Tooth/C-PXI/Labeled/Image",             # Capital I, singular
    "child_msk_dir":    "STS-2D-Tooth/C-PXI/Labeled/Mask",

    # ── Output ─────────────────────────────────────────────────────────────
    "output_dir":       "/kaggle/working/phase1_outputs",
    "best_model_name":  "swin_unetr_teacher_best.pth",

    # ── Image settings ─────────────────────────────────────────────────────
    # Swin window_size=(7,7) → spatial_size must be divisible by
    # window_size * 2^(num_downsamples) = 7*8=56 in each dim.
    # 640 / 56 = 11.4  →  use 672x336 (both divisible by 56)
    # Alternatively keep 640x320 — MONAI pads internally; both work.
    "spatial_size":     (640, 320),       # H x W
    "in_channels":      1,                # grayscale

    # ── Intensity windowing (dental HU equivalent for PNG) ─────────────────
    "a_min": -100,
    "a_max": 1500,

    # ── Training hyperparams ────────────────────────────────────────────────
    "seed":             42,
    "val_fraction":     0.15,             # 15% of labeled set → validation
    "epochs":           60,
    "batch_size":       1,                # keep 1 for T4 safety
    "accum_steps":      8,                # effective batch = 8
    "lr":               2e-4,
    "weight_decay":     1e-5,
    "lr_patience":      5,                # ReduceLROnPlateau patience
    "lr_factor":        0.5,

    # ── Swin UNETR architecture ─────────────────────────────────────────────
    "feature_size":     24,               # base feature dim (48 → OOM on T4)
    "depths":           (2, 2, 2, 2),     # transformer depth per stage
    "num_heads":        (3, 6, 12, 24),   # attention heads per stage
    "drop_rate":        0.0,
    "attn_drop_rate":   0.0,
    "dropout_path_rate":0.1,
    "use_checkpoint":   True,             # gradient checkpointing → saves ~30% VRAM

    # ── Misc ────────────────────────────────────────────────────────────────
    "num_workers":      2,
    "cache_rate":       1.0,              # cache all 900 pairs in RAM
    "conf_threshold":   0.95,             # used in Phase 2
}

# ─────────────────────────────────────────────────────────────────────────────
# 2.  SETUP
# ─────────────────────────────────────────────────────────────────────────────
set_determinism(seed=CFG["seed"])
os.makedirs(CFG["output_dir"], exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log.info(f"Device: {device}")
print_config()


# ─────────────────────────────────────────────────────────────────────────────
# 3.  DATA LOADING  — build image/mask path pairs
# ─────────────────────────────────────────────────────────────────────────────
def _gather_pairs(img_dir: str, msk_dir: str) -> list[dict]:
    """Match image PNGs to their corresponding mask PNGs by stem name."""
    root = Path(CFG["data_root"])
    imgs = sorted(glob.glob(str(root / img_dir / "*.png")))
    pairs = []
    for img_path in imgs:
        stem = Path(img_path).stem
        # Masks may have same name or suffixed with _mask — try both
        for suffix in ["", "_mask"]:
            msk_path = str(root / msk_dir / f"{stem}{suffix}.png")
            if os.path.exists(msk_path):
                pairs.append({"image": img_path, "label": msk_path})
                break
    log.info(f"  Found {len(pairs)} pairs in {img_dir}")
    return pairs


def build_data_dicts() -> tuple[list, list]:
    """
    Gather all labeled pairs (adult + child), shuffle, split 85/15.
    Returns (train_dicts, val_dicts).
    """
    all_pairs = _gather_pairs(CFG["labeled_img_dir"], CFG["labeled_msk_dir"])

    # Optionally add child PXI
    if CFG.get("child_img_dir"):
        child = _gather_pairs(CFG["child_img_dir"], CFG["child_msk_dir"])
        all_pairs += child

    random.seed(CFG["seed"])
    random.shuffle(all_pairs)

    n_val  = max(1, int(len(all_pairs) * CFG["val_fraction"]))
    n_train = len(all_pairs) - n_val
    train_dicts = all_pairs[:n_train]
    val_dicts   = all_pairs[n_train:]

    log.info(f"Train pairs: {len(train_dicts)} | Val pairs: {len(val_dicts)}")
    return train_dicts, val_dicts


# ─────────────────────────────────────────────────────────────────────────────
# 4.  MONAI TRANSFORMS
# ─────────────────────────────────────────────────────────────────────────────
# ── Shared base (applied to both train and val) ─────────────────────────────
BASE_TRANSFORMS = [
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),

    # ── CRITICAL MEMORY SAVER: RGB PNG → single-channel grayscale ──────────
    # LoadImaged loads PNG as (3, H, W); we keep only the luminance channel.
    # Using a Lambda transform inline:
    monai.transforms.Lambdad(
        keys=["image"],
        func=lambda x: x[[0], ...]   # R channel ≈ luminance for grayscale X-rays
    ),

    # ── Intensity windowing  ───────────────────────────────────────────────
    ScaleIntensityRanged(
        keys=["image"],
        a_min=CFG["a_min"], a_max=CFG["a_max"],
        b_min=0.0,          b_max=1.0,
        clip=True,
    ),

    # ── Resize to standard dental resolution ──────────────────────────────
    Resized(
        keys=["image", "label"],
        spatial_size=CFG["spatial_size"],
        mode=("bilinear", "nearest"),
    ),

    # ── Ensure label is binary (0/1 float) ─────────────────────────────────
    monai.transforms.Lambdad(
        keys=["label"],
        func=lambda x: (x > 0.5).float()
    ),

    # ── Z-score normalization ──────────────────────────────────────────────
    NormalizeIntensityd(keys=["image"], nonzero=True, channel_wise=True),

    EnsureTyped(keys=["image", "label"], dtype=torch.float32),
]

# ── Training augmentations (on top of base) ─────────────────────────────────
AUGMENTATIONS = [
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),       # horizontal flip
    RandRotate90d(keys=["image", "label"], prob=0.2, max_k=1),
    RandShiftIntensityd(keys=["image"], offsets=0.05, prob=0.3),
    RandGaussianNoised(keys=["image"], std=0.02, prob=0.3),
    RandAdjustContrastd(keys=["image"], gamma=(0.8, 1.2), prob=0.3),
]

train_transforms = Compose(BASE_TRANSFORMS + AUGMENTATIONS)
val_transforms   = Compose(BASE_TRANSFORMS)


# ─────────────────────────────────────────────────────────────────────────────
# 5.  DATASETS & DATALOADERS
# ─────────────────────────────────────────────────────────────────────────────
def build_loaders(train_dicts, val_dicts):
    train_ds = CacheDataset(
        data=train_dicts,
        transform=train_transforms,
        cache_rate=CFG["cache_rate"],
        num_workers=CFG["num_workers"],
    )
    val_ds = CacheDataset(
        data=val_dicts,
        transform=val_transforms,
        cache_rate=CFG["cache_rate"],
        num_workers=CFG["num_workers"],
    )

    train_loader = DataLoader(
        train_ds, batch_size=CFG["batch_size"],
        shuffle=True, num_workers=0,   # 0 workers when CacheDataset is used
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=1,
        shuffle=False, num_workers=0,
        pin_memory=True,
    )
    return train_loader, val_loader


# ─────────────────────────────────────────────────────────────────────────────
# 6.  MODEL — Swin UNETR (2D)
# ─────────────────────────────────────────────────────────────────────────────
def build_model() -> SwinUNETR:
    """
    2D Swin UNETR.
    img_size must be (H, W) tuple matching spatial_size.
    feature_size=24 is the T4-safe sweet spot (48 → OOM).
    use_checkpoint=True enables gradient checkpointing inside transformers.
    """
    model = SwinUNETR(
        #img_size       = CFG["spatial_size"],          # (640, 320)
        in_channels    = CFG["in_channels"],           # 1
        out_channels   = 1,                            # binary seg
        spatial_dims   = 2,                            # 2D mode
        feature_size   = CFG["feature_size"],          # 24
        depths         = CFG["depths"],
        num_heads      = CFG["num_heads"],
        drop_rate      = CFG["drop_rate"],
        attn_drop_rate = CFG["attn_drop_rate"],
        dropout_path_rate = CFG["dropout_path_rate"],
        use_checkpoint = CFG["use_checkpoint"],
    ).to(device)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log.info(f"Swin UNETR parameters: {n_params:,}")
    return model


# ─────────────────────────────────────────────────────────────────────────────
# 7.  LOSS, OPTIMIZER, SCHEDULER, METRICS
# ─────────────────────────────────────────────────────────────────────────────
def build_training_components(model):
    """
    DiceCELoss:
      - sigmoid=True because SwinUNETR returns raw logits (no sigmoid at end).
      - This avoids double-squashing that was the bug in the original R2U-Net code.
    """
    loss_fn = DiceCELoss(sigmoid=True, lambda_dice=0.6, lambda_ce=0.4)

    optimizer = AdamW(
        model.parameters(),
        lr           = CFG["lr"],
        weight_decay = CFG["weight_decay"],
    )

    scheduler = ReduceLROnPlateau(
        optimizer,
        mode       = "max",    # maximize Dice
        patience   = CFG["lr_patience"],
        factor     = CFG["lr_factor"],
        #verbose    = True,
    )

    # ── Metrics (applied AFTER sigmoid + threshold) ────────────────────────
    dice_metric = DiceMetric(
        include_background=False,   # teeth are foreground; bg excluded
        reduction="mean",
        get_not_nans=False,
    )
    hd95_metric = HausdorffDistanceMetric(
        include_background=False,
        percentile=95,
        reduction="mean",
        get_not_nans=True,
    )

    # Post-processing: sigmoid → round to 0/1
    post_pred  = Compose([Activationsd(keys="pred",  sigmoid=True),
                          AsDiscreted  (keys="pred",  threshold=0.5)])
    post_label = Compose([AsDiscreted  (keys="label", threshold=0.5)])

    return loss_fn, optimizer, scheduler, dice_metric, hd95_metric, post_pred, post_label


# ─────────────────────────────────────────────────────────────────────────────
# 8.  TRAINING LOOP — Phase 1 (Teacher)
# ─────────────────────────────────────────────────────────────────────────────
def train_one_epoch(model, loader, loss_fn, optimizer, scaler, epoch):
    model.train()
    optimizer.zero_grad(set_to_none=True)   # set_to_none saves a tiny bit of memory

    epoch_loss  = 0.0
    accum_steps = CFG["accum_steps"]

    for step, batch in enumerate(loader, start=1):
        imgs   = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        with autocast():
            preds = model(imgs)                     # raw logits, shape (B,1,H,W)
            loss  = loss_fn(preds, labels) / accum_steps

        scaler.scale(loss).backward()

        if step % accum_steps == 0 or step == len(loader):
            # Gradient clipping prevents exploding gradients in transformers
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        epoch_loss += loss.item() * accum_steps

        # ── Aggressive VRAM cleanup every 10 steps ─────────────────────────
        del imgs, labels, preds
        if step % 10 == 0:
            gc.collect()
            torch.cuda.empty_cache()

    avg_loss = epoch_loss / len(loader)
    return avg_loss


@torch.no_grad()
def validate(model, loader, dice_metric, hd95_metric, post_pred, post_label):
    """
    Run inference on validation set, compute Dice and HD95.
    Uses autocast for speed; no gradient tracking.
    """
    model.eval()
    dice_metric.reset()
    hd95_metric.reset()

    for batch in loader:
        imgs   = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        with autocast():
            preds = model(imgs)          # raw logits

        # Decollate batch → list of single-item dicts for post-processing
        batch_list = decollate_batch({"pred": preds, "label": labels})
        preds_post  = [post_pred (item) for item in batch_list]
        labels_post = [post_label(item) for item in batch_list]

        pred_tensors  = torch.stack([x["pred"]  for x in preds_post])
        label_tensors = torch.stack([x["label"] for x in labels_post])

        dice_metric(y_pred=pred_tensors, y=label_tensors)
        hd95_metric(y_pred=pred_tensors, y=label_tensors)

        del imgs, labels, preds, pred_tensors, label_tensors

    mean_dice           = dice_metric.aggregate().item()
    hd95_val, not_nans  = hd95_metric.aggregate()
    mean_hd95           = hd95_val.item() if not_nans.item() > 0 else float("inf")

    gc.collect()
    torch.cuda.empty_cache()
    return mean_dice, mean_hd95


# ─────────────────────────────────────────────────────────────────────────────
# 9.  MAIN — orchestrate everything
# ─────────────────────────────────────────────────────────────────────────────
def main():
    log.info("=" * 70)
    log.info("  PHASE 1: TEACHER TRAINING — Swin UNETR (2D)")
    log.info("=" * 70)

    # ── Data ─────────────────────────────────────────────────────────────
    train_dicts, val_dicts = build_data_dicts()
    train_loader, val_loader = build_loaders(train_dicts, val_dicts)

    # ── Model ────────────────────────────────────────────────────────────
    model = build_model()

    # ── Training components ───────────────────────────────────────────────
    (loss_fn, optimizer, scheduler,
     dice_metric, hd95_metric,
     post_pred, post_label) = build_training_components(model)

    scaler = GradScaler()           # AMP gradient scaler

    # ── Training state ────────────────────────────────────────────────────
    best_dice    = -1.0
    best_epoch   = 0
    history      = []               # for optional plotting / CSV export
    best_model_path = os.path.join(CFG["output_dir"], CFG["best_model_name"])

    log.info(f"Starting training for {CFG['epochs']} epochs...")

    for epoch in range(1, CFG["epochs"] + 1):
        # ── Train ─────────────────────────────────────────────────────────
        avg_loss = train_one_epoch(
            model, train_loader, loss_fn, optimizer, scaler, epoch
        )

        # ── Validate ──────────────────────────────────────────────────────
        mean_dice, mean_hd95 = validate(
            model, val_loader,
            dice_metric, hd95_metric,
            post_pred, post_label
        )

        # ── Scheduler step (maximize Dice) ─────────────────────────────────
        scheduler.step(mean_dice)
        current_lr = optimizer.param_groups[0]["lr"]

        # ── Log ───────────────────────────────────────────────────────────
        log.info(
            f"Epoch {epoch:03d}/{CFG['epochs']} | "
            f"Loss: {avg_loss:.4f} | "
            f"Val Dice: {mean_dice:.4f} | "
            f"Val HD95: {mean_hd95:.2f} px | "
            f"LR: {current_lr:.2e}"
        )

        history.append({
            "epoch": epoch, "loss": avg_loss,
            "dice": mean_dice, "hd95": mean_hd95, "lr": current_lr,
        })

        # ── Save best checkpoint ───────────────────────────────────────────
        if mean_dice > best_dice:
            best_dice  = mean_dice
            best_epoch = epoch
            torch.save({
                "epoch":      epoch,
                "model_state_dict":     model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scaler_state_dict":    scaler.state_dict(),
                "best_dice":  best_dice,
                "best_hd95":  mean_hd95,
                "cfg":        CFG,
            }, best_model_path)
            log.info(f"  ✓ Best model saved (Dice: {best_dice:.4f})")

        # ── Early stopping (optional) ──────────────────────────────────────
        # Uncomment if you want to add patience-based early stopping:
        # if epoch - best_epoch > 20:
        #     log.info("Early stopping triggered.")
        #     break

    # ── Training complete ─────────────────────────────────────────────────
    log.info("=" * 70)
    log.info(f"Training complete. Best Dice: {best_dice:.4f} at epoch {best_epoch}")
    log.info(f"Best model saved to: {best_model_path}")
    log.info("=" * 70)

    # ── Save history CSV ──────────────────────────────────────────────────
    import csv
    history_path = os.path.join(CFG["output_dir"], "training_history.csv")
    with open(history_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["epoch","loss","dice","hd95","lr"])
        writer.writeheader()
        writer.writerows(history)
    log.info(f"Training history saved to: {history_path}")

    # ── Quick summary for Phase 2 ─────────────────────────────────────────
    log.info("\n" + "="*50)
    log.info("  PHASE 2 INSTRUCTIONS (Pseudo-Labeling)")
    log.info("="*50)
    log.info(f"  Load checkpoint: {best_model_path}")
    log.info(f"  Use CFG['conf_threshold'] = {CFG['conf_threshold']}")
    log.info("  Run inference on unlabeled set, save masks where")
    log.info("  sigmoid(logit).max() > threshold.")
    log.info("="*50)

    return best_model_path, history


# ─────────────────────────────────────────────────────────────────────────────
# 10.  PHASE 2 STUB — Pseudo-Labeling (run after Phase 1)
# ─────────────────────────────────────────────────────────────────────────────
def run_pseudo_labeling(
    best_model_path: str,
    unlabeled_img_dir: str,
    pseudo_mask_dir: str,
    max_images: int = 20_000,      # start with subset; scale up later
):
    """
    Phase 2: Load Teacher → generate pseudo-masks for unlabeled images.
    Only saves masks where max pixel confidence > CFG['conf_threshold'].

    Args:
        best_model_path   : path to best Phase-1 checkpoint
        unlabeled_img_dir : directory with raw unlabeled PNGs
        pseudo_mask_dir   : where to write accepted pseudo-masks
        max_images        : cap on how many unlabeled images to process
    """
    os.makedirs(pseudo_mask_dir, exist_ok=True)

    # ── Load model ────────────────────────────────────────────────────────
    model = build_model()
    ckpt  = torch.load(best_model_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    log.info(f"Teacher loaded from epoch {ckpt['epoch']} (Dice: {ckpt['best_dice']:.4f})")

    # ── Unlabeled image list ──────────────────────────────────────────────
    all_imgs = sorted(glob.glob(os.path.join(unlabeled_img_dir, "*.png")))[:max_images]
    log.info(f"Running pseudo-labeling on {len(all_imgs)} images...")

    infer_transforms = Compose(BASE_TRANSFORMS)   # no augmentation for inference

    accepted = 0
    rejected = 0

    for img_path in all_imgs:
        data  = infer_transforms({"image": img_path, "label": img_path})   # label slot unused
        img_t = data["image"].unsqueeze(0).to(device)   # (1,1,H,W)

        with torch.no_grad(), autocast():
            logit = model(img_t)
            prob  = torch.sigmoid(logit).squeeze()       # (H,W)

        # Confidence gate: max pixel probability > threshold
        if prob.max().item() < CFG["conf_threshold"]:
            rejected += 1
            del img_t, logit, prob
            continue

        # Save binary pseudo-mask as PNG (0/255)
        mask_np  = (prob > 0.5).cpu().numpy().astype(np.uint8) * 255
        stem     = Path(img_path).stem
        out_path = os.path.join(pseudo_mask_dir, f"{stem}_pseudo.png")

        from PIL import Image
        Image.fromarray(mask_np).save(out_path)
        accepted += 1

        del img_t, logit, prob
        if accepted % 500 == 0:
            gc.collect(); torch.cuda.empty_cache()
            log.info(f"  Accepted: {accepted} | Rejected: {rejected}")

    log.info(f"Pseudo-labeling done. Accepted: {accepted} | Rejected: {rejected}")
    log.info(f"Pseudo-masks saved to: {pseudo_mask_dir}")
    return accepted


# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # ── PHASE 1 ───────────────────────────────────────────────────────────
    best_model_path, history = main()

    # ── PHASE 2 (uncomment after Phase 1 completes) ───────────────────────
    # unlabeled_dir  = os.path.join(CFG["data_root"], "STS-2D-Tooth/A-PXI/Unlabeled/images")
    # pseudo_dir     = "/kaggle/working/pseudo_masks"
    # run_pseudo_labeling(best_model_path, unlabeled_dir, pseudo_dir, max_images=20_000)

<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
2026-02-28 17:34:39.070731: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772300079.258716      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772300079.312323      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772300079.760715      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772300079.760743      55 computation_placer.cc:1

MONAI version: 1.5.2
Numpy version: 2.0.2
Pytorch version: 2.9.0+cu126
MONAI flags: HAS_EXT = False, USE_COMPILED = False, USE_META_DICT = False
MONAI rev id: d18565fb3e4fd8c556707f91ac280a2dc3f681c1
MONAI __file__: /usr/local/lib/python3.12/dist-packages/monai/__init__.py

Optional dependencies:
Pytorch Ignite version: 0.5.3
ITK version: NOT INSTALLED or UNKNOWN VERSION.
Nibabel version: 5.3.3
scikit-image version: 0.25.2
scipy version: 1.16.3
Pillow version: 11.3.0
Tensorboard version: 2.19.0
gdown version: 5.2.1
TorchVision version: 0.24.0+cu126
tqdm version: 4.67.1
lmdb version: NOT INSTALLED or UNKNOWN VERSION.
psutil version: 5.9.5
pandas version: 2.3.3
einops version: 0.8.1
transformers version: 5.2.0
mlflow version: NOT INSTALLED or UNKNOWN VERSION.
pynrrd version: NOT INSTALLED or UNKNOWN VERSION.
clearml version: NOT INSTALLED or UNKNOWN VERSION.

For details about installing the optional dependencies, please visit:
    https://docs.monai.io/en/latest/installation.html#instal

2026-02-28 17:34:51,964 | INFO | ======================================================================
2026-02-28 17:34:51,965 | INFO |   PHASE 1: TEACHER TRAINING — Swin UNETR (2D)
2026-02-28 17:34:51,965 | INFO | ======================================================================
2026-02-28 17:34:51,985 | INFO |   Found 850 pairs in STS-2D-Tooth/A-PXI/Labeled/Image
2026-02-28 17:34:51,988 | INFO |   Found 50 pairs in STS-2D-Tooth/C-PXI/Labeled/Image
2026-02-28 17:34:51,989 | INFO | Train pairs: 765 | Val pairs: 135
Loading dataset: 100%|██████████| 135/135 [00:01<00:00, 76.31it/s]
2026-02-28 17:35:04,439 | INFO | Swin UNETR parameters: 6,302,203
2026-02-28 17:35:04,442 | INFO | Starting training for 60 epochs...
2026-02-28 17:37:37,489 | INFO | Epoch 001/60 | Loss: 0.3935 | Val Dice: 0.8566 | Val HD95: 13.80 px | LR: 2.00e-04
2026-02-28 17:37:37,634 | INFO |   ✓ Best model saved (Dice: 0.8566)
2026-02-28 17:40:11,099 | INFO | Epoch 002/60 | Loss: 0.2334 | Val Dice: 0.8836 | Val H